# Treino do modelo

## 0. Configs

### 0.1 Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import warnings
import json
import sys

from pathlib import Path
from joblib import dump

# lib para Thompson Sampling
from mabwiser.mab import MAB, LearningPolicy

# sklearn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# funções utils
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.utils_eda import months, days, faixa_etaria, sep_milhar

# configurações de exibição
pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')

### 0.2 Base de dados

In [3]:
df = pd.read_parquet('../data/trusted/tabela_analitica.parquet', engine = 'pyarrow')
df = df.reset_index()
df['index'] = round(df['index'] / df['index'].max(), 3)

df.head()

,index,age,job,marital,education,faixa_etaria,housing,loan,contact,month,year,day_of_week,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y
0,0.0,56,housemaid,married,basic.4y,Entre 50 e 60 anos,no,no,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,0.0,57,services,married,high.school,Entre 50 e 60 anos,no,no,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,0.0,37,services,married,high.school,Entre 31 e 40 anos,yes,no,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,0.0,40,admin.,married,basic.6y,Entre 31 e 40 anos,no,no,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,0.0,56,services,married,high.school,Entre 50 e 60 anos,no,yes,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## 1. Baseline

Variáveis em que foram observadas maior conversão:
- Faixa etária: mais de 60 anos e menos de 25 anos
- Profissão: estudante e aposentado
- Contato: celular
- Resultado da campanha anterior: success

In [4]:
df_baseline = df[['faixa_etaria', 'job', 'contact', 'poutcome', 'y']].copy()

# baseline 1 (todas as características)
df_baseline['baseline1'] = np.where(
    (
        ((df_baseline['faixa_etaria'] == 'Mais de 60 anos') | (df_baseline['faixa_etaria'] == 'Até 25 anos')) &
        ((df_baseline['job'] == 'retired') | (df_baseline['job'] == 'student')) &
        (df_baseline['contact'] == 'cellular') &
        (df_baseline['poutcome'] == 'success')
    ),
    'yes',
    'no'
)

# baseline 2 (pelo menos 1 característica)
df_baseline['baseline2'] = np.where(
    (
        ((df_baseline['faixa_etaria'] == 'Mais de 60 anos') | (df_baseline['faixa_etaria'] == 'Até 25 anos')) |
        ((df_baseline['job'] == 'retired') | (df_baseline['job'] == 'student')) |
        (df_baseline['contact'] == 'cellular') |
        (df_baseline['poutcome'] == 'success')
    ),
    'yes',
    'no'
)

# baseline 3 (pelo menos 1 característica - removendo forma de contato)
df_baseline['baseline3'] = np.where(
    (
        ((df_baseline['faixa_etaria'] == 'Mais de 60 anos') | (df_baseline['faixa_etaria'] == 'Até 25 anos')) |
        ((df_baseline['job'] == 'retired') | (df_baseline['job'] == 'student')) |
        # (df_baseline['contact'] == 'cellular') |
        (df_baseline['poutcome'] == 'success')
    ),
    'yes',
    'no'
)

df_baseline


,faixa_etaria,job,contact,poutcome,y,baseline1,baseline2,baseline3
0,Entre 50 e 60 anos,housemaid,telephone,nonexistent,no,no,no,no
1,Entre 50 e 60 anos,services,telephone,nonexistent,no,no,no,no
2,Entre 31 e 40 anos,services,telephone,nonexistent,no,no,no,no
3,Entre 31 e 40 anos,admin.,telephone,nonexistent,no,no,no,no
4,Entre 50 e 60 anos,services,telephone,nonexistent,no,no,no,no
...,...,...,...,...,...,...,...,...
41183,Mais de 60 anos,retired,cellular,nonexistent,yes,no,yes,yes
41184,Entre 41 e 50 anos,blue-collar,cellular,nonexistent,no,no,yes,no
41185,Entre 50 e 60 anos,retired,cellular,nonexistent,no,no,yes,yes
41186,Entre 41 e 50 anos,technician,cellular,nonexistent,yes,no,yes,no


In [68]:
print('Conversão geral')
df_baseline['y'].value_counts(normalize = True)

Conversão geral


y
no     0.887346
yes    0.112654
Name: proportion, dtype: float64

In [35]:
df_output = df_baseline[['y']].value_counts().reset_index().rename(columns = {'y':'output', 'count':'y'})

for i in range(1, 4):
    tmp = df_baseline[[f'baseline{i}']].value_counts().reset_index().rename(columns = {f'baseline{i}':'output', 'count':f'baseline{i}'})
    df_output = df_output.merge(tmp, on = 'output', how = 'outer')

df_output

,output,y,baseline1,baseline2,baseline3
0,no,36548,41007,13859,36152
1,yes,4640,181,27329,5036


In [5]:
def calcular_metricas_baseline(baseline, titulo):

    vp = df_baseline[(df_baseline['y'] == 'yes') & (df_baseline[baseline] == 'yes')].shape[0]
    fp = df_baseline[(df_baseline['y'] == 'no') & (df_baseline[baseline] == 'yes')].shape[0]
    vn = df_baseline[(df_baseline['y'] == 'no') & (df_baseline[baseline] == 'no')].shape[0]
    fn = df_baseline[(df_baseline['y'] == 'yes') & (df_baseline[baseline] == 'no')].shape[0]

    acuracia = (vp + vn) / (vp + fp + vn + fn)
    recall = (vp) / (vp + fn)
    precision = vp / (vp + fp)
    f1_score = 2 * (precision * recall) / (precision + recall)

    print(f"{baseline} | {titulo}\n{'-' * 100}\n")

    print(f"Total de observações:  {sep_milhar(vp + fp + vn + fn)}\n")
    print(f"Verdadeiros positivos: {sep_milhar(vp)}")
    print(f"Falsos positivos:      {sep_milhar(fp)}")
    print(f"Verdadeiros negativos: {sep_milhar(vn)}")
    print(f"Falsos negativos:      {sep_milhar(fn)}\n")
    print(f"Acurácia:              {round(acuracia, 4)}")
    print(f"Recall:                {round(recall, 4)} (vp / tudo que realmente é positivo)")
    print(f"Precision:             {round(precision, 4)} (vp / tudo que foi classificado positivo)")
    print(f"F1-score:              {round(f1_score, 4)} (média harmônica entre precision e recall)")

In [65]:
calcular_metricas_baseline(baseline = 'baseline1', titulo = 'Cliente tem todas as características')

baseline1 | Cliente tem todas as características
----------------------------------------------------------------------------------------------------

Total de observações:  41.188

Verdadeiros positivos: 136
Falsos positivos:      45
Verdadeiros negativos: 36.503
Falsos negativos:      4.504

Acurácia:              0.8896
Recall:                0.0293 (vp / tudo que realmente é positivo)
Precision:             0.7514 (vp / tudo que foi classificado positivo)
F1-score:              0.0564 (média harmônica entre precision e recall)


In [66]:
calcular_metricas_baseline(baseline = 'baseline2', titulo = 'Cliente tem pelo menos 1 das características')

baseline2 | Cliente tem pelo menos 1 das características
----------------------------------------------------------------------------------------------------

Total de observações:  41.188

Verdadeiros positivos: 4.012
Falsos positivos:      23.317
Verdadeiros negativos: 13.231
Falsos negativos:      628

Acurácia:              0.4186
Recall:                0.8647 (vp / tudo que realmente é positivo)
Precision:             0.1468 (vp / tudo que foi classificado positivo)
F1-score:              0.251 (média harmônica entre precision e recall)


In [67]:
calcular_metricas_baseline(baseline = 'baseline3' , titulo = 'Cliente tem pelo menos 1 das características (removendo forma de contato)')

baseline3 | Cliente tem pelo menos 1 das características (removendo forma de contato)
----------------------------------------------------------------------------------------------------

Total de observações:  41.188

Verdadeiros positivos: 1.623
Falsos positivos:      3.413
Verdadeiros negativos: 33.135
Falsos negativos:      3.017

Acurácia:              0.8439
Recall:                0.3498 (vp / tudo que realmente é positivo)
Precision:             0.3223 (vp / tudo que foi classificado positivo)
F1-score:              0.3355 (média harmônica entre precision e recall)


Conclusões do baseline:

- Como a conversão geral é de 11% (target muito desbalanceada), falar que ninguém vai converter já fornece uma acurácia de 89%, então não é uma boa métrica para ser acompanhada - olhar também precision e recall
- Apesar do baseline 1 ter uma acurácia melhor, o recall ficou muito baixo, o que fez o f1 score ficar muito baixo também
- O baseline 2 teve acurária de 42% (o que já é muito ruim), e a precision ficou ruim
- Para o baselne 3, precision e recall ficaram baixos, mas melhor que os demais baselines. Cabe ao modelo superar essas métricas
- O melhor baseline é o 3:
    - Job: retired ou student
    - Faixa etária: até 25 ou mais de 60
    - Resultado da campanha anterior: success

## 2. Treino do modelo

### 2.1 Separar em treino, validação e teste

In [6]:
# separação por fraçöFileExistsError
# train = df[df['index'] <= 0.7]
# valid = df[(df['index'] > 0.7) & (df['index'] <= 0.85)]
# test = df[df['index'] > 0.85]

# separação por ano
train = df[df['year'] == 2008]
valid = df[df['year'] == 2009]
test = df[df['year'] == 2010]

#### Proporções

In [22]:
tamanho_inicial = df.shape[0]

print(f"Tamanho inicial:    {sep_milhar(tamanho_inicial)}\n")

print(f"- Treino (2008):    {sep_milhar(train.shape[0])} ({round(train.shape[0] * 100 / tamanho_inicial, 1)}%)")
print(f"- Validação (2009): {sep_milhar(valid.shape[0])} ({round(valid.shape[0] * 100 / tamanho_inicial, 1)}%)")
print(f"- Teste (2010):     {sep_milhar(test.shape[0])} ({round(test.shape[0] * 100 / tamanho_inicial, 1)}%)")

Tamanho inicial:    41.188

- Treino (2008):    27.690 (67.2%)
- Validação (2009): 11.440 (27.8%)
- Teste (2010):     2.058 (5.0%)


In [23]:
for tb in [train, valid, test]:
    print(tb['y'].value_counts(normalize = True))

y
no     0.951643
yes    0.048357
Name: proportion, dtype: float64
y
no     0.805245
yes    0.194755
Name: proportion, dtype: float64
y
yes    0.52138
no     0.47862
Name: proportion, dtype: float64


In [24]:
for tb in [train, valid, test]:
    print(tb['contact'].value_counts(normalize = True))

contact
cellular     0.504731
telephone    0.495269
Name: proportion, dtype: float64
contact
cellular     0.917657
telephone    0.082343
Name: proportion, dtype: float64
contact
cellular     0.811467
telephone    0.188533
Name: proportion, dtype: float64


In [70]:
with pd.option_context('display.float_format', '{:.2f}'.format):
    display(train[['month']].value_counts(normalize=True).reset_index(name = 'treino_2008')\
    .merge(valid[['month']].value_counts(normalize=True).reset_index(name = 'valid_2009'), on = 'month', how = 'outer')\
    .merge(test[['month']].value_counts(normalize=True).reset_index(name = 'test_2010'), on = 'month', how = 'outer'))


,month,treino_2008,valid_2009,test_2010
0,03. mar,NaN,0.02,0.13
1,04. apr,NaN,0.21,0.08
2,05. may,0.28,0.51,0.10
3,06. jun,0.16,0.06,0.11
4,07. jul,0.24,0.02,0.15
5,08. aug,0.19,0.07,0.11
6,09. sep,NaN,0.02,0.15
7,10. oct,0.00,0.04,0.10
8,11. nov,0.13,0.03,0.06
9,12. dec,0.00,0.02,NaN


### 2.2 Pré-processamento

- One-hot encoding das categóricas
- Padronização das numéricas
- Contact é o braço, year é apenas split e faixa_etaria é derivada de age

Como o LinTS trabalha com uma representação numérica do contexto, as variáveis categóricas são transformadas em one-hot encoding, enquanto as variáveis numéricas são padronizadas.

O pré-processor é ajustado exclusivamente nos dados de treino e posteriormente aplicado à validação e ao teste, evitando vazamento de informação entre os períodos

In [22]:
df.head()

,index,age,job,marital,education,faixa_etaria,housing,loan,contact,month,year,day_of_week,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y
0,0.0,56,housemaid,married,basic.4y,Entre 50 e 60 anos,no,no,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,0.0,57,services,married,high.school,Entre 50 e 60 anos,no,no,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,0.0,37,services,married,high.school,Entre 31 e 40 anos,yes,no,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,0.0,40,admin.,married,basic.6y,Entre 31 e 40 anos,no,no,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,0.0,56,services,married,high.school,Entre 50 e 60 anos,no,yes,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [9]:
numeric_features = ["age", "previous", "emp_var_rate", "cons_price_idx", "cons_conf_idx", "euribor3m", "nr_employed"]

categorical_features = ["job", "marital", "education", "housing", "loan", "month", "day_of_week", "poutcome"]

context_features = numeric_features + categorical_features

preprocessor = ColumnTransformer([("num", StandardScaler(), numeric_features), ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)])

X_train = np.asarray(preprocessor.fit_transform(train[context_features]), dtype=np.float64)
X_valid = np.asarray(preprocessor.transform(valid[context_features]), dtype=np.float64)
X_test = np.asarray(preprocessor.transform(test[context_features]), dtype=np.float64)

a_train = train["contact"].astype(str).to_numpy()
a_valid = valid["contact"].astype(str).to_numpy()
a_test = test["contact"].astype(str).to_numpy()

y_train = train["y"].map({"no": 0, "yes": 1}).astype(int).to_numpy()
y_valid = valid["y"].map({"no": 0, "yes": 1}).astype(int).to_numpy()
y_test = test["y"].map({"no": 0, "yes": 1}).astype(int).to_numpy()

arms = sorted(train["contact"].unique().tolist())

feature_names = preprocessor.get_feature_names_out().tolist()

print("Treino:   ", X_train.shape)
print("Validação:", X_valid.shape)
print("Teste:    ", X_test.shape)
print("Braços:   ", arms)

Treino:    (27690, 52)
Validação: (11440, 52)
Teste:     (2058, 52)
Braços:    ['cellular', 'telephone']


### 2.3 Definir hiperparâmetros

- Otimização de alpha e l2_lambda no período de validação, usando reward estimado P(y=1 | contexto, braço) como critério principal.

A validação de hiperparâmetros utiliza exclusivamente os dados de 2008 para treinamento e os dados de 2009 para validação, mantenndo 2010 isolado para validação final

Serão testados diferentes valores de alpha (responsável por controlar a intensidade da exploração no LinTS) e l2_lambda (associado à regularização do modelo linear)

Como a base histórica não contém o resultado de todos os braços para cada cliente, um modelo auxiliar de recompensa estima P(y=1 | contexto, braço) e permite comparar as configurações no conjunto de validação

Diferentes seeds são utilizadas para reduzir a chance de selecionar uma configuração apenas por uma realização aleatória favorável

In [ ]:
def treinar_modelos_recompensa(X, actions, rewards, arms):
    models = {}
    for arm in arms:
        mask = actions == arm
        model = LogisticRegression(max_iter=2000, solver="lbfgs", random_state=42)
        model.fit(X[mask], rewards[mask])
        models[arm] = model
    return models

reward_models_train = treinar_modelos_recompensa(X_train, a_train, y_train, arms)

reward_valid_por_braco = {arm: reward_models_train[arm].predict_proba(X_valid)[:, 1] for arm in arms}

alphas = [0.10, 0.25, 0.50, 1.00, 2.00]
l2_lambdas = [0.10, 1.00, 10.00]
seeds = [42, 123, 2026]

tuning_rows = []

for alpha in alphas:
    for l2_lambda in l2_lambdas:
        for seed in seeds:
            model = MAB(arms=arms, learning_policy=LearningPolicy.LinTS(alpha=alpha, l2_lambda=l2_lambda, scale=False), seed=seed, n_jobs=-1)
            model.fit(decisions=a_train, rewards=y_train, contexts=X_train)

            recommended = np.asarray(model.predict(contexts=X_valid)).reshape(-1)
            estimated_reward = np.array([reward_valid_por_braco[action][i] for i, action in enumerate(recommended)])

            matched = recommended == a_valid
            observed_reward = y_valid[matched].mean() if matched.any() else np.nan

            tuning_rows.append({"alpha": alpha, "l2_lambda": l2_lambda, "seed": seed, "reward_estimado": estimated_reward.mean(), 
                                "reward_observado_matches": observed_reward, "match_rate": matched.mean(), "n_matches": matched.sum()})

tuning_runs_df = pd.DataFrame(tuning_rows)

tuning_summary_df = tuning_runs_df\
    .groupby(["alpha", "l2_lambda"], as_index=False)\
    .agg(reward_estimado_medio=("reward_estimado", "mean"), 
         reward_estimado_std=("reward_estimado", "std"), 
         reward_observado_matches_medio=("reward_observado_matches", "mean"), 
         match_rate_medio=("match_rate", "mean"))

tuning_summary_df = tuning_summary_df.sort_values(["reward_estimado_medio", "match_rate_medio"], ascending=[False, False]).reset_index(drop=True)

best_alpha = float(tuning_summary_df.loc[0, "alpha"])
best_l2_lambda = float(tuning_summary_df.loc[0, "l2_lambda"])

display(tuning_summary_df)
print("Melhor alpha:", best_alpha)
print("Melhor l2_lambda:", best_l2_lambda)

,alpha,l2_lambda,reward_estimado_medio,reward_estimado_std,reward_observado_matches_medio,match_rate_medio
0,0.10,10.0,0.664931,0.001399,0.167199,0.800408
1,0.25,10.0,0.637310,0.001537,0.169391,0.768211
2,0.50,10.0,0.552956,0.001214,0.177465,0.674971
3,0.10,1.0,0.522848,0.001014,0.160763,0.645775
4,1.00,10.0,0.482708,0.001111,0.183938,0.596096
5,0.25,1.0,0.452402,0.000426,0.176665,0.566550
6,2.00,10.0,0.440028,0.001227,0.188633,0.550524
7,0.10,0.1,0.433375,0.001828,0.180871,0.544697
8,0.50,1.0,0.423518,0.002607,0.184516,0.533275
9,0.25,0.1,0.410578,0.002794,0.189091,0.519755


Melhor alpha: 0.1
Melhor l2_lambda: 10.0


### 2.4 Treino final do LinTS

- Treino + validação e avaliação exclusivamente no teste de 2010.

Após a seleção dos hiperparâmetros, os períodos de treino e validação são reunidos e o LinTs final é treinado com os dados de 2008 e 2009, deixando 2010 para teste.

- O contexto corresponde às características disponíveis do cliente antes de cada decisão
- `contact` representa os braços `cellular` e `telephone`
- a variável `y` é transformada em recompensa binária, sendo 1 para conversão e 0 para não conversão

O Thompson Sampling realiza exploração a partir da incerteza associada às estimativas de recompensa, permitindo que a política explore alternativas incertas ao mesmo tempo em que tende a explorar mais frequentemente ações que apresentam evidências de melhor desempenho

In [13]:
train_valid = pd.concat([train, valid], ignore_index=True).sort_values("index").reset_index(drop=True)

preprocessor_final = ColumnTransformer([("num", StandardScaler(), numeric_features), ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)])

X_train_valid = np.asarray(preprocessor_final.fit_transform(train_valid[context_features]), dtype=np.float64)
X_test_final = np.asarray(preprocessor_final.transform(test[context_features]), dtype=np.float64)

a_train_valid = train_valid["contact"].astype(str).to_numpy()
y_train_valid = train_valid["y"].map({"no": 0, "yes": 1}).astype(int).to_numpy()

reward_models_final = treinar_modelos_recompensa(X_train_valid, a_train_valid, y_train_valid, arms)

lints = MAB(arms=arms, learning_policy=LearningPolicy.LinTS(alpha=best_alpha, l2_lambda=best_l2_lambda, scale=False), seed=42, n_jobs=-1)
lints.fit(decisions=a_train_valid, rewards=y_train_valid, contexts=X_train_valid)

recommended_test = np.asarray(lints.predict(contexts=X_test_final)).reshape(-1)

reward_test_por_braco = {arm: reward_models_final[arm].predict_proba(X_test_final)[:, 1] for arm in arms}

estimated_reward_lints = np.array([reward_test_por_braco[action][i] for i, action in enumerate(recommended_test)])

matched_test = recommended_test == a_test

historical_arm_reward = pd.DataFrame({"arm": a_train_valid, "reward": y_train_valid}).groupby("arm")["reward"].mean()
baseline_arm = historical_arm_reward.idxmax()

recommended_baseline = np.repeat(baseline_arm, len(test))
estimated_reward_baseline = reward_test_por_braco[baseline_arm]

eval_test = pd.DataFrame({"index": test["index"].to_numpy(), "observed_action": a_test, "recommended_action": recommended_test, "reward": y_test, "matched": matched_test, "estimated_reward_lints": estimated_reward_lints, "baseline_action": recommended_baseline, "estimated_reward_baseline": estimated_reward_baseline})

eval_test["observed_reward_lints"] = np.where(eval_test["matched"], eval_test["reward"], np.nan)

print("Braço do baseline:", baseline_arm)
print("Reward histórico por braço:")
display(historical_arm_reward)

Braço do baseline: cellular
Reward histórico por braço:


arm
cellular     0.118084
telephone    0.046193
Name: reward, dtype: float64

### 2.5 Métricas finais

As principais métricas da política serão organizadas em um dataframe para facilitar a comparação e rastreabilidade.

Serão armazenados indicadores como:
- match rate
- reward observado nos casos em que a recomendação coincide com a ação histórica
- reward estimado pelo modelo auxiliar
- distribuição das recomendações entre os braços
- lift estimado em relação ao baseline

In [ ]:
# 6. Métricas principais do LinTS e persistência da avaliação.

results_dir = Path("../models/results")
results_dir.mkdir(parents=True, exist_ok=True)

reward_lints_estimado = eval_test["estimated_reward_lints"].mean()
reward_baseline_estimado = eval_test["estimated_reward_baseline"].mean()
lift_absoluto = reward_lints_estimado - reward_baseline_estimado
lift_relativo = lift_absoluto / reward_baseline_estimado if reward_baseline_estimado > 0 else np.nan

metricas_lints = pd.DataFrame([{"modelo": "LinTS", "split": "test_2010", "alpha": best_alpha, "l2_lambda": best_l2_lambda, 
                                "n_observacoes": len(eval_test), "n_matches": int(eval_test["matched"].sum()), "match_rate": eval_test["matched"].mean(), 
                                "reward_observado_matches": eval_test.loc[eval_test["matched"], "reward"].mean(), "reward_estimado_model_based": reward_lints_estimado, 
                                "baseline_arm": baseline_arm, "reward_estimado_baseline": reward_baseline_estimado, 
                                "lift_absoluto_estimado": lift_absoluto, "lift_relativo_estimado": lift_relativo, 
                                "pct_cellular": (eval_test["recommended_action"] == "cellular").mean(), "pct_telephone": (eval_test["recommended_action"] == "telephone").mean()}])

metricas_lints.to_csv(results_dir / "metricas_lints.csv", index=False)
eval_test.to_csv(results_dir / "avaliacao_lints_test.csv", index=False)
tuning_runs_df.to_csv(results_dir / "tuning_runs_lints.csv", index=False)
tuning_summary_df.to_csv(results_dir / "tuning_summary_lints.csv", index=False)

print(metricas_lints.T)

                                     0
modelo                           LinTS
split                        test_2010
alpha                              0.1
l2_lambda                         10.0
n_observacoes                     2058
n_matches                         1455
match_rate                    0.706997
reward_observado_matches      0.562199
reward_estimado_model_based     0.7109
baseline_arm                  cellular
reward_estimado_baseline      0.740607
lift_absoluto_estimado       -0.029707
lift_relativo_estimado       -0.040112
pct_cellular                  0.857629
pct_telephone                 0.142371


### 2.6 Comparação com baseline

- O baseline 3 é classificatório e o LinTS escolhe braços; 
- accuracy, precision, recall e F1 ficam como referência, enquanto a comparação de política usa reward e lift.

O baseline originalmente construído é uma regra de classificação de propensão à conversão, podendo ser avaliado por accuracy, precision, recall e f1-score

O LinTS é uma política de decisão, cuja função é selecionar um braço para cada contexto

A comparação principal entre políticas utiliza reward, match rate e lift estimado, enquanto as métricas classificatórias do baseline são mantidas como referência complementar

Como politica estática de comparação para o bandit, utiliza-se sempre o braço que apresentou maior reward histórico nos dados disponíveis para treinamento

In [ ]:
baseline3_test_pred = np.where((test["age"] < 25) | (test["age"] > 60) | test["job"].isin(["student", "retired"]) | (test["poutcome"] == "success"), 1, 0)

baseline3_test_metricas = pd.DataFrame([{"amostra": "base_completa_resultado_anterior", "accuracy": 0.8439, "recall": 0.3498, "precision": 0.3223, "f1": 0.3355}, {"amostra": "test_2010", "accuracy": accuracy_score(y_test, baseline3_test_pred), "recall": recall_score(y_test, baseline3_test_pred, zero_division=0), "precision": precision_score(y_test, baseline3_test_pred, zero_division=0), "f1": f1_score(y_test, baseline3_test_pred, zero_division=0)}])

baseline_match = a_test == baseline_arm

comparacao_politicas = pd.DataFrame([{"politica": f"Baseline estático: sempre {baseline_arm}", "n": len(test), "match_rate": baseline_match.mean(), "reward_observado_matches": y_test[baseline_match].mean(), "reward_estimado_model_based": reward_baseline_estimado, "lift_absoluto_vs_baseline": 0.0, "lift_relativo_vs_baseline": 0.0}, {"politica": "LinTS", "n": len(test), "match_rate": matched_test.mean(), "reward_observado_matches": y_test[matched_test].mean(), "reward_estimado_model_based": reward_lints_estimado, "lift_absoluto_vs_baseline": lift_absoluto, "lift_relativo_vs_baseline": lift_relativo}])

baseline3_test_metricas.to_csv(results_dir / "baseline3_classificacao.csv", index=False)
comparacao_politicas.to_csv(results_dir / "comparacao_politicas.csv", index=False)

print("Baseline classificatório:")
display(baseline3_test_metricas)

print("\nComparação correta entre políticas:")
display(comparacao_politicas)

Baseline classificatório:


,amostra,accuracy,recall,precision,f1
0,base_completa_resultado_anterior,0.84390,0.349800,0.322300,0.335500
1,test_2010,0.61759,0.624418,0.635674,0.629995



Comparação correta entre políticas:


,politica,n,match_rate,reward_observado_matches,reward_estimado_model_based,lift_absoluto_vs_baseline,lift_relativo_vs_baseline
0,Baseline estático: sempre cellular,2058,0.811467,0.576647,0.740607,0.000000,0.000000
1,LinTS,2058,0.706997,0.562199,0.710900,-0.029707,-0.040112


### 2.7 Comparação por braço

A avaliação por braço permite analisar como o comportamento e o desempenho estimado do LinTS se distribuem entre cellular e telephone, em vez de observar apenas uma métrica agregada

Para cada braço recomendado pelo modelo, são calculados:
- o número e a proporção de recomendações
- o reward estimado
- o reward da política baseline para os mesmos contextos
- o lift estimado
- o match rate com as ações históricas
- o reward observado nos casos de correspondência

Esta análise ajuda a verificar se um eventual ganho agregado está concentrado em apenas um braço ou se a política contextual encontra situações em que diferentes ações são potencialmente vantajosas

In [ ]:
comparacao_bracos_rows = []

for arm in arms:
    mask = eval_test["recommended_action"] == arm

    if mask.sum() == 0:
        continue

    reward_lints_arm = eval_test.loc[mask, "estimated_reward_lints"].mean()
    reward_baseline_arm = eval_test.loc[mask, "estimated_reward_baseline"].mean()
    lift_arm_abs = reward_lints_arm - reward_baseline_arm
    lift_arm_rel = lift_arm_abs / reward_baseline_arm if reward_baseline_arm > 0 else np.nan

    matched_arm = mask & eval_test["matched"]

    comparacao_bracos_rows.append({"braco_recomendado": arm, "n_recomendacoes": int(mask.sum()), "pct_recomendacoes": mask.mean(), "reward_estimado_lints": reward_lints_arm, "reward_estimado_baseline": reward_baseline_arm, "lift_absoluto_estimado": lift_arm_abs, "lift_relativo_estimado": lift_arm_rel, "match_rate": eval_test.loc[mask, "matched"].mean(), "reward_observado_matches": eval_test.loc[matched_arm, "reward"].mean() if matched_arm.any() else np.nan})

comparacao_bracos = pd.DataFrame(comparacao_bracos_rows)
comparacao_bracos.to_csv(results_dir / "comparacao_por_braco.csv", index=False)

display(comparacao_bracos)

,braco_recomendado,n_recomendacoes,pct_recomendacoes,reward_estimado_lints,reward_estimado_baseline,lift_absoluto_estimado,lift_relativo_estimado,match_rate,reward_observado_matches
0,cellular,1765,0.857629,0.730690,0.73069,0.000000,0.00000,0.802266,0.571328
1,telephone,293,0.142371,0.591691,0.80035,-0.208659,-0.26071,0.133106,0.230769


### 2.8 Desempenho por variável

A análise segmentada serve para verificar o comportamento da política em diferentes grupos de clientes

Para cada categoria, serão comparados o reward estimado do LinTS e o reward estimado da política baseline, além da distribuição dos braços recomendados e do match rate com o histórico

Essa análise é importante para um contextual bandit, pois permite verificar se o modelo está efetivamente produzindo diferentes decisões conforme o contexto, em vez de simplesmente aprender a recomendar o mesmo braço para praticamente todos os clientes

In [23]:
segment_variables = ["job", "faixa_etaria", "education", "poutcome", "marital", "month"]

segment_base = pd.concat([test[segment_variables].reset_index(drop=True), eval_test[["recommended_action", "reward", "matched", "estimated_reward_lints", "estimated_reward_baseline"]].reset_index(drop=True)], axis=1)

segment_rows = []

for variable in segment_variables:
    for category, group in segment_base.groupby(variable, dropna=False):
        reward_lints_segment = group["estimated_reward_lints"].mean()
        reward_baseline_segment = group["estimated_reward_baseline"].mean()
        lift_abs_segment = reward_lints_segment - reward_baseline_segment
        lift_rel_segment = lift_abs_segment / reward_baseline_segment if reward_baseline_segment > 0 else np.nan
        matched_group = group[group["matched"]]

        segment_rows.append({"variavel": variable, "categoria": category, "n": len(group), "reward_estimado_lints": reward_lints_segment, "reward_estimado_baseline": reward_baseline_segment, "lift_absoluto_estimado": lift_abs_segment, "lift_relativo_estimado": lift_rel_segment, "match_rate": group["matched"].mean(), "reward_observado_matches": matched_group["reward"].mean() if len(matched_group) > 0 else np.nan, "pct_cellular_lints": (group["recommended_action"] == "cellular").mean(), "pct_telephone_lints": (group["recommended_action"] == "telephone").mean()})

comparacao_segmentos = pd.DataFrame(segment_rows)
comparacao_segmentos = comparacao_segmentos.sort_values(["variavel", "lift_absoluto_estimado"], ascending=[True, False]).reset_index(drop=True)

comparacao_segmentos.to_csv(results_dir / "comparacao_por_segmento.csv", index=False)

display(comparacao_segmentos)

,variavel,categoria,n,reward_estimado_lints,reward_estimado_baseline,lift_absoluto_estimado,lift_relativo_estimado,match_rate,reward_observado_matches,pct_cellular_lints,pct_telephone_lints
0,education,basic.6y,33,0.764599,0.772207,-0.007608,-0.009853,0.696970,0.695652,0.939394,0.060606
1,education,unknown,162,0.739047,0.762880,-0.023833,-0.031241,0.765432,0.564516,0.895062,0.104938
2,education,basic.9y,123,0.677860,0.702764,-0.024905,-0.035438,0.699187,0.523256,0.886179,0.113821
3,education,university.degree,788,0.717377,0.743303,-0.025926,-0.034879,0.737310,0.554217,0.859137,0.140863
4,education,professional.course,296,0.722642,0.753860,-0.031218,-0.041411,0.682432,0.559406,0.851351,0.148649
5,education,basic.4y,201,0.700106,0.732745,-0.032639,-0.044543,0.731343,0.612245,0.875622,0.124378
6,education,high.school,455,0.691829,0.730799,-0.038970,-0.053325,0.641758,0.554795,0.824176,0.175824
7,faixa_etaria,Entre 31 e 40 anos,626,0.714649,0.739683,-0.025034,-0.033844,0.709265,0.549550,0.884984,0.115016
8,faixa_etaria,Até 25 anos,200,0.736510,0.764110,-0.027600,-0.036121,0.680000,0.588235,0.860000,0.140000
9,faixa_etaria,Entre 26 e 30 anos,385,0.734551,0.762541,-0.027990,-0.036706,0.709091,0.509158,0.854545,0.145455


### 2.9 Persistência do modelo e demais artefatos

Salvar todos os componentes necessários para reproduzir a inferência fora do notebook
- LinTS treinado
- Pipeline de pré-processamento
- Modelos auxiliares de recompensa
- Lista de features
- Braços disponíveis
- Metadados do experimento

Os artefatos podem ser carregados posteriormente pela API, garantindo que uma nova observação passe exatamente pelo mesmo processamento utilizado durante o treinamento

Os metatados também registram informações como hiperparâmetros, anos utilizados, definição da recompensa e metodologia de avaliação offline, aumentando a rastreabilidade do experimento

In [ ]:
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

final_feature_names = preprocessor_final.get_feature_names_out().tolist()

metadata = {
    "model_type": "Linear Thompson Sampling",
    "library": "mabwiser",
    "arms": arms,
    "baseline_arm": baseline_arm,
    "alpha": best_alpha,
    "l2_lambda": best_l2_lambda,
    "seed": 42,
    "train_years": [2008, 2009],
    "test_year": 2010,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "context_features": context_features,
    "excluded_from_context": ["index", "year", "contact", "y", "faixa_etaria"],
    "reward_definition": {"no": 0, "yes": 1},
    "offline_evaluation": "historical matching + model-based reward estimation",
    "causal_interpretation": False
}

bundle = {
    "preprocessor": preprocessor_final,
    "bandit": lints,
    "reward_models": reward_models_final,
    "arms": arms,
    "baseline_arm": baseline_arm,
    "context_features": context_features,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "feature_names_after_preprocessing": final_feature_names,
    "metadata": metadata
}

dump(lints, models_dir / "lints_model.joblib")
dump(preprocessor_final, models_dir / "preprocessor.joblib")
dump(reward_models_final, models_dir / "reward_models.joblib")
dump(bundle, models_dir / "lints_bundle.joblib")

with open(models_dir / "lints_metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=4)

pd.DataFrame({"feature": final_feature_names}).to_csv(models_dir / "lints_features.csv", index=False)

print("Artefatos persistidos.")

Artefatos persistidos.
